# Wavesurfer-Video-DLC Alignment

Align wavesurfer acquisition data with video frames via sync signal.

Default behavior: loads existing alignment templates from `alignment_templates/cam2/` on MICROSCOPE (legacy location). Recomputes only if missing or `FORCE_RECOMPUTE=True`.

Outputs go to `stroke_orofacial_pipeline_outputs/alignment/{cam_name}/` on MICROSCOPE, with per-camera separation:

- `alignment/{cam_name}/templates/{animal}/{YYYYMMDD}.npz` — alignment template
- `alignment/{cam_name}/params/{animal}/{YYYYMMDD}.json` — per-session params + run log

**Phase 6.7 update:** orchestration now flows through `SessionManifest`. Cell 3 builds one manifest per (animal, date); cell 4 passes each manifest to `load_or_compute_alignment()`. Phase 3 is DLC-agnostic by design — it relates wavesurfer to camera frames; DLC isn't read here, so sessions without DLC inference still align cleanly.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None to process all dates in the
# animal's session range (from configs/animals.yaml).
DATES = None

# Camera index (cam2 = DLC camera)
CAM_IDX = 2

# Overwrite mode: "skip" (default), "prompt", or "force"
OVERWRITE = "skip"

# Force recompute even if existing template is available
FORCE_RECOMPUTE = False

In [ ]:
import datetime
import json
import logging

import numpy as np

from stroke_orofacial.alignment.timebase import load_or_compute_alignment
from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.manifest import build_manifest
from stroke_orofacial.paths import PathResolver

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
align_params = defaults["alignment"]
align_params["cam_idx"] = CAM_IDX

output_root = resolver.pipeline_outputs("alignment")
cam_name = f"cam{CAM_IDX}"

In [ ]:
# ============================================================
# Build a SessionManifest per (animal, date).
# Surfaces wavesurfer + cam2 video discovery in the notebook state
# before alignment runs. Phase 3 only needs wavesurfer + camera CSV;
# DLC isn't read here.
# ============================================================

manifests = []  # list of (animal_id, date, manifest)

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for date in dates:
        manifest = build_manifest(animal_id, date, resolver, info)
        manifests.append((animal_id, date, manifest))

n_with_template = sum(1 for _, _, m in manifests if m.alignment_template is not None)
n_with_ws = sum(1 for _, _, m in manifests if m.wavesurfer_file is not None)
n_with_video = sum(1 for _, _, m in manifests if m.video_files)
print(f"Built {len(manifests)} manifests:")
print(f"  with existing alignment template ({cam_name}): {n_with_template}")
print(f"  with wavesurfer file:                          {n_with_ws}")
print(f"  with cam{CAM_IDX} video files:                          {n_with_video}")

In [ ]:
# ============================================================
# Align each session by passing its manifest to load_or_compute_alignment().
# Save per-camera template + params.json under stroke_orofacial_pipeline_outputs/.
# ============================================================

n_loaded, n_computed, n_skipped, n_error = 0, 0, 0, 0

for animal_id, date, manifest in manifests:
    date_str = date.strftime("%Y%m%d")
    out_path = output_root / cam_name / "templates" / animal_id / f"{date_str}.npz"

    if OVERWRITE == "skip" and out_path.exists():
        n_skipped += 1
        continue

    try:
        result = load_or_compute_alignment(
            manifest, resolver, align_params,
            force_recompute=FORCE_RECOMPUTE,
        )

        if result.source == "loaded":
            n_loaded += 1
        else:
            n_computed += 1

        out_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(
            str(out_path),
            sig_camIdx__idx_ws=result.sig_camIdx__idx_ws,
            sig_wsIdx__idx_cam=result.sig_wsIdx__idx_cam,
            slope_ws_per_camFrame=result.slope_ws_per_camFrame,
            intercept_ws_samples=result.intercept_ws_samples,
            Fs_ws_input=result.Fs_ws,
            source=result.source,
            allow_pickle=False,
        )

        params_path = output_root / cam_name / "params" / animal_id / f"{date_str}.json"
        params_path.parent.mkdir(parents=True, exist_ok=True)
        record = {
            "session_id": f"{animal_id}_{date_str}",
            "animal": animal_id,
            "date": date.isoformat(),
            "cam_name": cam_name,
            "run_timestamp": datetime.datetime.now().isoformat(),
            "skipped": False,
            "params": align_params,
            "source": result.source,
            "processing_log": [],
        }
        with open(params_path, "w") as f:
            json.dump(record, f, indent=2, default=str)

    except (FileNotFoundError, ValueError) as e:
        n_error += 1
        logging.warning("Skipping %s/%s: %s", animal_id, date_str, e)

print(f"\nLoaded: {n_loaded}, Computed: {n_computed}, Skipped: {n_skipped}, Errors: {n_error}")